# Datamine SELEXY Process Example

This notebook demonstrates how to configure and run the **`selexy`** process wrapper in `dmstudio`.

## Process Description

## SELEXY

Selectively copy records for which X and Y co-ordinate values lie within a perimeter defined in a perimeter file.

If the parameter @**OUTSIDE** =1, then values outside the perimeter will be selected. A perimeter must consist of at least the standard explicit numeric fields **XP, YP, ZP, PVALUE** and **PTN**. It must contain more than 2 and less than 1500 points.

### Multiple perimeters

Multiple perimeters may be processed at once. The output file will contain records which are selected by any of the perimeters. If selection is inside perimeters (default) then the effect is that selected records will come from the union of all perimeters. If selection is outside perimeters, then records will be selected if they lie outside any one perimeter, even if they lie inside another.

There is a limit to the number of perimeters which may be processed at one time. Under no circumstances may more than 100 perimeters be processed; but this limit may be lower, depending on the implementation and the number of points in each perimeter. All perimeters are first read into the real part of Studio 3's virtual array. Each point occupies 2 words. The size of this array is usually 100,000 words for implementations. With these values, the maximum number of points is 50,000 for most implementations. Each perimeter is closed in the array, by addition of an extra point. Thus the number of perimeters that can be handled depends on the total number of points in the input file. Any perimeters that cannot fit in their entirety into the virtual array will be ignored with a warning.

Once the perimeters have been read in, **SELEXY** makes a single pass through the input file, examining each record against all perimeters. This is a very fast method of processing data, and ensures that the order of the output file is the same as the order of the input file.

**Note** : The X,Y fields in the &**IN** file are assumed to refer to the same co-ordinate system as the **XP,YP** fields in the &**PERIM** file.

### Single Perimeter

A single perimeter may be selected from a multi-perimeter file by setting the required **PVALUE** into optional parameter @**PERIM**.

### Flagging selected records

Selected records may be flagged by up to 4 words of information taken from the perimeter that selected them. These four words may be either 4 numeric fields, or a combination of alphanumeric and numeric fields up to 4 words long. These fields are identified as * **ATTRIB1** -* **ATTRIB4** ; they must exist in the perimeter file, and will be placed in the output file if they do not already exist. For example, the **PVALUE** field may be used to flag which perimeter the record was selected by. If a record should be selected by more than one perimeter, then the values will be taken from the last perimeter that selected the record.

The value of the * **ATTRIB** fields used in flagging will be that associated with the first point of the perimeter.

### In Place operations

The input and output files may be the same for in-place flagging of values. Retrieval criteria may also be used if required. There is however no point in in-place selection unless the input file contains a field that may be flagged to indicate selection. In-place updating is specified by setting the input and output files to the same names, and specifying at least one * **ATTRIB** field that exists in both the perimeter file and the input file. The **PVALUE** field is often used for this purpose. An error is generated if the *ATTRIB field does not exist already in the input file, as new fields cannot be added in-place.

### Input Files:

* **in** (Undefined):
  Input file for selection. Must have explicit numeric fields X and Y.
  Required=Yes

* **perim** (String):
  Perimeter file to control selection. Must have perimeter points in clockwise order,
  with the perimeter closed. The fields required are **XP,YP,ZP,PTN** , and **PVALUE**
  (standard perimeter format). All perimeters in the file will be used, up to a maximum of
  100, or the number that will fit into the real part of the virtual array (usually
  100,000 for all except your application on the PC, where it is 10,000). Perimeters which
  do not fit will be ignored with a warning. May also contain fields **ATTRIB1** -**4**
  which can be carried across to the output file. The value of these fields at the first
  point is used.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing all records lying within (or optionally outside) the perimeter.
  The **OUT** file may be the same as the IN file for in-place operations, unless extra
  fields ( **ATTRIB1** -**4**) from the perimeter file are being added.
  Required=Yes

### Fields:

* **x** (Numeric : IN):
  Field in **IN** file defining the X co-ordinate.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN):
  Field in **IN** file defining the Y co-ordinate.
  Default=Undefined
  Required=Yes

* **attribs** (Undefined : Undefined):
  Field from the perimeter file to be placed into the output file for all records which are selected. Up to 4 words may be entered, which may be 4 numeric fields or a mixture of alphanumeric and numeric fields totalling 4 words.
  Default=Undefined
  Required=No

### Parameters:

* **outside**:

* **Options** (1: Copies records of a file which have X and Y co-ordinates lying outside the):
  perimeter (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **perim**:
  Set to the required **PVALUE** field to select a particular perimeter from **PERIM**.
  If **PERIM** is not set, then all perimeters will be processed.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **print**:
  >=1 Display summary stats for each perimeter and DDs.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('selexy')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx",
    "_vb_holes.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute selexy
print("Running selexy...")
dm_cmd.selexy(
    in_i='t_assays',  # required
    perim_i='t_assays',  # required
    out_o='t_selexy_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    perim_p='optional',  # required
    # attribs_f=['optional'],  # optional
    # outside_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("selexy execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_selexy_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")